# Preparação dos Dados

## Objetivo

Nesta etapa realizamos o pré-processamento do dataset Insurance Cost Dataset.

As atividades incluem:

- Carregamento dos dados
- Análise inicial
- Tratamento das variáveis categóricas
- Separação entre variáveis de entrada (X) e variável alvo (y)
- Normalização dos atributos
- Salvamento dos dados preparados para as próximas etapas

In [1]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler

## Carregamento do Dataset

O dataset contém informações de clientes de seguro médico, incluindo idade, sexo, IMC, número de filhos, hábito de fumar e região.

A variável alvo é:

- charges → custo do seguro médico

In [2]:
df = pd.read_csv('../data/insurance.csv')  

print("Shape:", df.shape)
print("\nPrimeiras linhas:")
df.head()

Shape: (1338, 7)

Primeiras linhas:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Análise Exploratória Inicial

Verificamos:

- Tipos de dados
- Valores ausentes
- Estatísticas descritivas

Essa etapa garante que o dataset esteja consistente antes do treinamento.

In [3]:
print(df.dtypes)
print("\nValores nulos:")
print(df.isnull().sum())
print("\nEstatísticas:")
df.describe()

age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object

Valores nulos:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Estatísticas:


,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [4]:
# Criação de uma cópia para preservar o dataset original
df_encoded = df.copy()

# LabelEncoder para sex e smoker
le = LabelEncoder()
df_encoded['sex'] = le.fit_transform(df['sex'])        # female=0, male=1
df_encoded['smoker'] = le.fit_transform(df['smoker'])  # no=0, yes=1

# One-Hot Encoding para a variável region
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

print("Colunas após encoding:")
print(df_encoded.columns.tolist())
df_encoded.head()

Colunas após encoding:
['age', 'sex', 'bmi', 'children', 'smoker', 'charges', 'region_northwest', 'region_southeast', 'region_southwest']


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


## Separação das Variáveis

A variável alvo (y) é o custo do seguro.

As demais variáveis compõem o conjunto de entrada (X).

In [5]:

y = df_encoded['charges']
X = df_encoded.drop(columns=['charges'])

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)
print("\nColunas de X:", X.columns.tolist())

Shape de X: (1338, 8)
Shape de y: (1338,)

Colunas de X: ['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']


## Normalização dos Dados

A clusterização utiliza distância entre pontos.

Como as variáveis possuem escalas diferentes
(ex.: idade varia em dezenas e charges em milhares),
aplicamos o StandardScaler para padronizar os atributos.

Após a transformação:

- média ≈ 0
- desvio padrão ≈ 1

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Antes da normalização:")
print(X.describe().round(2))

print("\nDepois da normalização:")
print(X_scaled.describe().round(2))

Antes da normalização:
           age      sex      bmi  children  smoker
count  1338.00  1338.00  1338.00   1338.00  1338.0
mean     39.21     0.51    30.66      1.09     0.2
std      14.05     0.50     6.10      1.21     0.4
min      18.00     0.00    15.96      0.00     0.0
25%      27.00     0.00    26.30      0.00     0.0
50%      39.00     1.00    30.40      1.00     0.0
75%      51.00     1.00    34.69      2.00     0.0
max      64.00     1.00    53.13      5.00     1.0

Depois da normalização:
           age      sex      bmi  children   smoker  region_northwest  \
count  1338.00  1338.00  1338.00   1338.00  1338.00           1338.00   
mean     -0.00    -0.00    -0.00     -0.00     0.00              0.00   
std       1.00     1.00     1.00      1.00     1.00              1.00   
min      -1.51    -1.01    -2.41     -0.91    -0.51             -0.57   
25%      -0.87    -1.01    -0.72     -0.91    -0.51             -0.57   
50%      -0.01     0.99    -0.04     -0.08    -0.51    

In [7]:
joblib.dump(scaler, '../data/scaler.pkl')
print("Scaler salvo!")

Scaler salvo!


In [8]:
X_scaled.to_csv('../data/X_scaled.csv', index=False)
y.to_csv('../data/y.csv', index=False)

print("Dados salvos com sucesso!")
print(f"X_scaled: {X_scaled.shape}")
print(f"y: {y.shape}")

Dados salvos com sucesso!
X_scaled: (1338, 8)
y: (1338,)


## Resultado da Etapa

Os dados preparados foram salvos para utilização nas próximas etapas:

- X_scaled.csv
- y.csv
- scaler.pkl

Esses arquivos serão utilizados na clusterização e regressão.